# Data Loading, Storage and File Formats
## DAT540 Introduction to Data Science
## University of Stavanger
### L08

In [13]:
import numpy as np
import pandas as pd
from pandas import json_normalize
from lxml import objectify

- File inputs and outputs with pandas
- Reading text and other formats from disk
- loading data from disk, databases and networks (web APIs)
- pandas data loading (read) instance methods

<img src='./images/loadfiles.png' width='450'>

- HDF5 package is a interface to the HDF5 binary data format. 
  - store huge amounts of numerical data, and easily manipulate that data from NumPy
  - can slice into multi-terabyte datasets stored on disk, as if they were real NumPy arrays
  - thousands of datasets can be stored in a single file, categorized and tagged
- msgpack is an efficient binary serialization format. It lets you exchange data among multiple languages 
- **pickle** module implements binary protocols for serializing and de-serializing a Python object structure
- sas files stored as either XPORT or SAS7BDAT format files.
- **feather** provides binary columnar serialization for data frames
  - It is designed to make reading and writing data frames efficient 
  - allows sharing data across different data analysis languages

- pandas read_* instance methods load and converts the data into a DataFrame
- Optional arguments for these functions may fall into few categories:
  - *Indexing* can treat one or more columns as row index from the returned DataFrame, and/or to determine column names from a row 
  - *Type interface and data conversion* includes the user-defined value conversions and custom list of missing value markers
  - *Datatime parsing* including combining capability, combining date and time information spread over multiple columns into a single column
  - *Iterating* support for iterating over chunks of very large files
  - *Unclean data issues* skipping rows or a footer, comments or others

- reading **CSVs** (read_csv)
- Read CSV (comma-separated) file into DataFrame

In [ ]:
# let us check the structure of file ex1.csv from filesystem
!head ./data/ex1.csv

In [ ]:
# using pd.read_csv to read this file directly will set the delimiter to comma (,)
# If the delimiter is different we have the pass the delimiter argument with the delimiting value
ex1 = pd.read_csv('./data/ex1.csv')
ex1

In [ ]:
# However, since this file doesnot have any header names we can ignore it by passing the argument header=None
ex1 = pd.read_csv('./data/ex1.csv', header=None)
ex1

In [ ]:
# If we rather want to manually name the columns we can use the names=[] argument (without the header argument)
ex1 = pd.read_csv('./data/ex1.csv', names=['zero', 'one', 'two', 'three', 'four'])
ex1

In [ ]:
# In order to set a column as the index for the rows, we can use the argument index_col.
# For multi/hierarchical index we pass a list of columns
# The columns can be indices or labels
ex1 = pd.read_csv('./data/ex1.csv', names=['zero', 'one', 'two', 'three', 'four'], index_col=['four'])
ex1

In [ ]:
! head ./data/ex2.csv

- How would you load this data set
- Is there a header
- What would be the delimiter

In [ ]:
# We would use regular expression as delimiter
# \s+ indicates one or more whitespaces
ex2 = pd.read_csv('data/ex2.csv', header=None, delimiter=r'\s+')
ex2

- Cases where a file might have rows that are not relevent to the overall dataset, can be skipped using argument *skiprow* list  of row indices

In [ ]:
!cat ./data/ex3.csv

In [ ]:
ex3 = pd.read_csv('./data/ex3.csv', skiprows=[0,1,3])
ex3

In [ ]:
!head ./data/titanic.csv

- How will you load the titanic.csv file into a dataframe
- Set hierarchical index on columns: Sex, Age
- Also drop column 'Unnamed: 0' if present

In [ ]:
titanic = pd.read_csv('./data/titanic.csv', delimiter=',', index_col=['Sex', 'Age'], comm)
titanic.drop(['Unnamed: 0'], axis=1, inplace=True)
titanic.head(3)

- Some of the frequently used options in pandas.read_csv

<img src='./images/read_csv.png' width=800, height=400>

- *Reading Text Files in Pieces*
- Used when processing very large files or figuring out the right set of arguments
- Also useful to load a section of the file, visualize top rows and figure out its structure
- The *nrows* argument can be supplied with an integer value to read only that many rows and not the whole file
- To read a file in pieces, the argument *chunksize* can be set as a number of rows 
  - A *TextParser* object is returned that allows iteration over the parts
  - A *get_chunk* method is also available to TextParser that enables pieces of arbitrary size to be read

In [ ]:
titanic = pd.read_csv('./data/titanic.csv', nrows=5)
titanic.shape

In [ ]:
# We can also read the data in chunks
titanic = pd.read_csv('./data/titanic.csv', chunksize=100, delimiter=',')
titanic
type(titanic)

In [ ]:
# Here we want to count the freq. of ages
age = pd.Series([], dtype=float)
for data in titanic:
  age = age.add(data['Age'].value_counts(), fill_value=0)

In [ ]:
print('size:', age.size)
age.head(10)

- *Writing Data to Text Format*
- Using a Series or DataFrame *to_csv(filename)* instance method, the data struct, can be written to disk.
- The default delimiter is a comma, however, it can be overridden using the *delimiter* argument
- *na_rep* argument can be used to denote missing values by some sentinel value rather than empty string
- By default, the row and column labels are written to disk. This can be overwritten using attributes *index=False* and/or *header=False*
- using *columns=[]* argument, columns can be filtered and ordered while writing to disk

In [ ]:
titanic = pd.read_csv('./data/titanic.csv', delimiter=',')

- Write the titanic object to disk with filename titanicmod.csv
- Suppress row labels/indices
- use comma as delimiter
- only select these columns in the given order [PassengerId, Age, Sex, Survived]

In [ ]:
titanic.to_csv('./data/titanicmod.csv', columns=['PassengerId', 'Age', 'Sex', 'Survived'], index=False)

In [ ]:
!head ./data/titanicmod.csv

- *Working with Delimited Formats*
- How will you convert the following file to a DataFrame

In [ ]:
!cat ./data/ex4.rtf

In [ ]:
# load/read the file as a table
ex4 = pd.read_table('./data/ex4.rtf', nrows=10, header=None)
ex4

In [ ]:
# load/read the file as a table and skip rows that are not required
rows_to_skip = list(range(9)) + [10]
print('Rows to skip:', rows_to_skip)
ex4 = pd.read_table('./data/ex4.rtf', header=None, skiprows=rows_to_skip)
ex4

In [ ]:
# Now let us load it as a csv
ex4 = pd.read_csv('./data/ex4.rtf', header=None, skiprows=rows_to_skip)
ex4

In [ ]:
# remove the special chars in col 4
ex4.replace(to_replace=r'\\*}*', value='', regex=True, inplace=True)
ex4

In [ ]:
# Now let us load it as a csv with the first row as header
ex4 = pd.read_csv('./data/ex4.rtf', header=0, skiprows=rows_to_skip)
# remove the special chars in col 4
ex4.replace(to_replace=r'\\*}*', value='', regex=True, inplace=True)
ex4

# How can we do replace the special char in the header without using replace on columns attribute

In [ ]:
# Now let us load it as a csv with header as None
ex4 = pd.read_csv('./data/ex4.rtf', header=None, skiprows=rows_to_skip)
# remove the special chars in col 4
ex4.replace(to_replace=r'\\*}*', value='', regex=True, inplace=True)
ex4


In [ ]:
# Set the proper row and col labels
ex4.columns = ex4.iloc[0]
ex4.drop(0, inplace=True)
ex4.set_index('ID', drop=True, inplace=True)
ex4

In [ ]:
ex4.columns

- *JSON (JavaScript Object Notation) Data*
- a standard format for sending data over HTTP requests 
```javascript
{'name': 'Wes',
'places_lived': ['US', 'NO', 'UK', 'DE'],
'pet': null,
'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['D', 'Zuko']},
             {'name': 'Katie', 'age': 38,
              'pets': ['Sixes', 'Stache', 'Cisco']}]}```

- The module *json* is the default library for operating on JSON file formats
- To convert a JSON string to Python, the *json.loads* method is used
- To convert a Python object back to JSON, *json.dump* method is used
- *pandas.read_json* can automatically convert JSON datasets into a Series or a DataFrame
  - it assumes that each object in the JSON array is a row in a table
- pandas can export data to JSON format using *pandas.to_json* method on a Series or a DataFrame

In [ ]:
import json

json_str = '''
{
    "name": "Wes",
    "places_lived": ["US", "NO", "UK", "DE"],
    "pet": null,
    "siblings": [
        {"name": "Scott", "age": 30, "pets": ["D", "Zuko"]},
        {"name": "Katie", "age": 38, "pets": ["Sixes", "Stache", "Cisco"]}
    ]
}
'''

parsed_json = json.loads(json_str) # parse the json string into a python dictionary
parsed_json



In [ ]:
print(json.dumps(parsed_json, indent=1))# convert back to json string with indentation

In [ ]:
# Reading, using pandas
pd.read_json('./data/jsonex.json')

- However, *read_json* only produces two rows in this case
  - This is because it assumes that each object in the JSON array is a row in a table
- Inorder to create a DataFrame with all elements as a cell we have to flatten the json object
- We use *json_normalize* for flattening json on pandas DataFrame

In [ ]:
json_normalize(pd.read_json('./data/jsonex.json').loc['maths'])

- *XML and HTML: Web Scraping*
- Multiple libraries are available for processing HTML & XML formats
  - lxml, BeautifulSoup, html5lib
- pandas has a build-in function *read_html* that uses lxml and Beautiful Soup to automatically parse tables out of HTML as DataFrames
- read_html takes both local and remote html files as input

In [ ]:
tables = pd.read_html('./data/html_tables.html')
print('Number of retrived tables:', len(tables))

- XML is another structured data format that is more generic than compared to HTML
- We load plant_catalog.xml as a DataFrame
- In order to load a xml format, we have to first import class *objectify* from module lxml
- We use to instance method *parse* from objectify to parse the xml file to lxml object
- next we iterate through the lxml object and create the relevant dataframe by choosing the required elements

In [ ]:
! head ./data/plant_catalog.xml

In [ ]:
obj = objectify.parse(open('./data/plant_catalog.xml'))
type(obj)

In [ ]:
# We store all the leaf nodes
data = []
# We choose not to select some properties of the each plant
skip_tag = ['AVAILABILITY']
# We get a generator yielding each root element CATALOG
root = obj.getroot()
# We go through each root element and get their child nodes

for element in root:  
  for child in element.getchildren():   
    leafnodes = {}
    for leaf in child.getchildren():        
      if leaf.tag not in skip_tag:
        leafnodes[leaf.tag] = leaf.text
    data.append(leafnodes)

df = pd.DataFrame(data)
df.head(2)

- *Binary Data Formats*
- Serialization is a mechanism to store data in binary format
- Python, provides a built-in module *pickle* for serialization
- pandas objects have a *to_pickle* instance method that writes data to disk in pickle format
- pickled data can be read from disk using the *pandas.read_pickle* method

In [ ]:
df.to_pickle('./data/plant_catalog')

- pandas also provides opportunities to read an Excel file
- The instance method *read_excel* can be used to read a excel file
  - along with the file path
  - a string or integer value can be provided as a second argument depicting the the Sheet Name or index
  - a list of string or integers can also be provided as a second argument to load multiple sheets
  - *None* is provided as the second argument to load all sheets
  - When a single value is provided, a DataFrame is returned
  - With None or list as the second argument, a *Dict* of DataFrames are returned having the keys representing the sheets
  - Inorder to write as excel, we first create an *pd.ExcelWriter* and then add DataFrames to the excel writer as sheets using instance method *to_excel(writer, SheetName)*

```python
writer = pd.ExcelWriter('./data/output.xlsx')
df1.to_excel(writer,'Sheet1')
df2.to_excel(writer,'Sheet2')
writer.save()
```

In [ ]:
df = pd.read_excel('./data/ex1.xlsx', sheet_name='Sheet1') # read excel file to dataframe
df